# 06 · Testing Beam Pipelines

> **Module 8 · Testing & Observability · Lecture 8.1**

In this notebook you will:

- Use the testing pyramid: most tests need no runner at all
- Test pure logic directly, with no Beam involved
- Drive a pipeline with `TestPipeline` and `assert_that`
- Meet the `use_global_window=False` problem and write a custom matcher for it
- Test a `CombineFn` directly, including its merge-order-independence property
- Build a deterministic stream with `TestStream`

---

## 1. The testing pyramid

Testing a Beam pipeline is not the same as testing ordinary Python, because
Beam defers almost everything until run time. A transform graph can be built
without error and still produce the wrong numbers.

The answer is a **pyramid**, not a single kind of test:

| Level | What it tests | Needs a runner? | Speed |
|-------|---------------|------------------|-------|
| **Unit** | Pure functions: parsing, validation, currency conversion | No | Milliseconds |
| **Component** | One `DoFn`, one `CombineFn`, one transform | No | Milliseconds |
| **Pipeline** | A whole graph: `TestPipeline` + `assert_that` | Yes, locally | Seconds |
| **Integration** | The real pipeline against real storage | Yes, and often GCP | Minutes |

Most of your tests belong at the bottom. The course project is deliberately
split that way: `project/transforms.py` has a pure half that runs with no
Apache Beam installed at all.

The setup below puts the project modules and the test helpers on `sys.path`.


In [ ]:
import sys
import itertools
from datetime import datetime
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = COURSE_ROOT.parent
PROJECT_DIR = COURSE_ROOT / "project"
TESTS_DIR = PROJECT_DIR / "tests"
for p in (str(PROJECT_DIR), str(TESTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import apache_beam as beam
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.util import assert_that, equal_to
from apache_beam.transforms.window import FixedWindows
from apache_beam.utils.timestamp import Timestamp

from conftest import beam_test_options
from transforms import (
    ParseOrderFn,
    RevenueByRegionFn,
    build_revenue_summary,
    is_valid_order,
    normalise_currency,
    parse_order_line,
    try_parse_order_line,
)

TEST_OPTS = beam_test_options()
print("apache-beam", beam.__version__)
print("test options ready")


## 2. Level 1 — pure logic, no runner

These are ordinary `assert` statements. They need no Beam, no pipeline, no
runner, and they run in milliseconds. If you have a hundred tests in a suite
and ninety of them are here, your feedback loop stays fast.

Note that `normalise_currency`, `parse_order_line` and `is_valid_order` live
in the *pure half* of `project/transforms.py` precisely so they can be tested
this way. That split is a design decision, not an accident.


In [ ]:
# Currency conversion.
assert normalise_currency(100.0, "EUR") == 100.0
assert normalise_currency(100.0, "usd") == 92.0     # lower case is accepted
assert normalise_currency(100.0, "GBP") == 117.0
assert normalise_currency(1000.0, "JPY") == 6.4

# An unknown currency is an error, not a silent zero.
try:
    normalise_currency(10.0, "XYZ")
    raise AssertionError("expected a ValueError for an unknown currency")
except ValueError as exc:
    print("unknown currency ->", exc)

# Parsing.
row = parse_order_line(
    "o1,c2,2026-01-15T00:00:00+00:00,EMEA,web,EUR,complete,10.5,p1"
)
assert row["order_id"] == "o1"
assert row["amount"] == 10.5
assert is_valid_order(row)

# A malformed line is rejected, not raised.
assert try_parse_order_line("not-an-order,missing-fields,garbage") is None
assert try_parse_order_line("") is None
assert not is_valid_order({**row, "region": "UNKNOWN_REGION"})
assert not is_valid_order({**row, "amount": -1.0})
assert not is_valid_order({**row, "status": "unknown-status"})

# The reference aggregation.
summary = build_revenue_summary([row], statuses=("complete",))
assert summary == {"EMEA": {"order_count": 1, "revenue_eur": 10.5}}, summary

print("all pure-logic assertions passed")


## 3. Level 2 — testing a `CombineFn` directly

A `CombineFn` is a plain Python object. You can drive it by hand, with no
runner at all — and you *should*, because the properties that make a combiner
scalable are exactly the properties a hand-driven test can check.

| Method | Contract |
|--------|----------|
| `create_accumulator()` | Returns a fresh, empty accumulator |
| `add_input(acc, element)` | Adds one element; returns the accumulator |
| `merge_accumulators(accs)` | Merges partial accumulators from different workers |
| `extract_output(acc)` | Turns the accumulator into the final output |

The critical property is **merge-order independence**: Beam merges partial
accumulators in an arbitrary order, so `merge(a, b)` must equal
`merge(b, a)`, and the whole result must be the same as adding the inputs one
at a time.


In [ ]:
combiner = RevenueByRegionFn()

# An empty accumulator extracts to zero, not to an error.
assert combiner.extract_output(combiner.create_accumulator()) == {
    "order_count": 0,
    "revenue_eur": 0.0,
}

ORDERS = [
    {"amount": 10.0, "currency": "EUR"},
    {"amount": 20.0, "currency": "EUR"},
    {"amount": 30.0, "currency": "EUR"},
]

# Merge two accumulators built from disjoint halves of the input.
left = combiner.create_accumulator()
for order in ORDERS[:2]:
    left = combiner.add_input(left, order)

right = combiner.create_accumulator()
for order in reversed(ORDERS[2:]):
    right = combiner.add_input(right, order)

merged = combiner.merge_accumulators([left, right])
print("merged from two halves:", combiner.extract_output(merged))
assert combiner.extract_output(merged) == {"order_count": 3, "revenue_eur": 60.0}

# The merge must not care about the order of the accumulators.
reversed_merge = combiner.merge_accumulators([right, left])
assert combiner.extract_output(reversed_merge) == combiner.extract_output(merged)

# And the whole thing must not care about the order of the inputs.
distinct = set()
for permutation in itertools.permutations(ORDERS):
    acc = combiner.create_accumulator()
    for order in permutation:
        acc = combiner.add_input(acc, order)
    distinct.add(tuple(sorted(combiner.extract_output(acc).items())))

print("distinct results over all", len(list(itertools.permutations(ORDERS))), "input orders:", len(distinct))
assert len(distinct) == 1, distinct

print("the combiner is commutative and associative")


## 4. Level 3 — `TestPipeline` and `assert_that`

Now we run an actual pipeline graph. `TestPipeline` is Beam's test harness:
it runs your pipeline on the local runner and, crucially, it **fails the test**
if any `assert_that` inside it does not hold.

The modern API is `beam.testing.test_pipeline.TestPipeline`. The older
`PTest` style is deprecated and has different semantics — do not mix them.

The pattern:

```python
with TestPipeline(options=beam_test_options()) as p:
    result = p | "Create" >> beam.Create(...) | "Transform" >> beam.Map(...)
    assert_that(result, equal_to([...]))
```

Two details worth knowing:

- `beam_test_options()` from `project/tests/conftest.py` sets
  `--allow_unsafe_triggers`, which Beam requires before it will accept
  `AfterCount` triggers.
- Two `assert_that` calls in the *same* pipeline need distinct `label=`
  arguments, or Beam raises `RuntimeError: A transform with label
  "[6]: assert_that" already exists in the pipeline.`


In [ ]:
ORDER_LINES = [
    "o1,c1,2026-01-15T10:00:00+00:00,EMEA,web,EUR,complete,100.00,p1",
    "o2,c2,2026-01-15T11:00:00+00:00,EMEA,web,USD,complete,200.00,p2",
    "o3,c3,2026-01-15T12:00:00+00:00,AMER,web,EUR,complete,50.00,p3",
    "o4,c4,2026-01-15T13:00:00+00:00,AMER,web,EUR,cancelled,999.00,p4",
    "o5,c5,2026-01-15T14:00:00+00:00,EMEA,web,GBP,complete,100.00,p5",
]


def _orders(lines, pipeline):
    """Build a PCollection of (region, order) pairs from raw CSV lines."""
    return (
        pipeline
        | "Create" >> beam.Create(lines)
        | "Parse" >> beam.ParDo(ParseOrderFn())
        | "KeyByRegion" >> beam.Map(lambda order: (order["region"], order))
    )


with TestPipeline(options=TEST_OPTS) as p:
    result = (
        _orders(ORDER_LINES, p)
        | "Revenue" >> beam.CombinePerKey(RevenueByRegionFn())
    )
    assert_that(
        result,
        equal_to([
            # EMEA = EUR 100 + USD 200 x 0.92 (184) + GBP 100 x 1.17 (117)
            ("EMEA", {"order_count": 3, "revenue_eur": 401.0}),
            # AMER = EUR 50 + cancelled EUR 999 — the combiner does no filtering.
            ("AMER", {"order_count": 2, "revenue_eur": 1049.0}),
        ]),
    )

print("the combiner test passed")


## 5. The `use_global_window=False` problem

Every `assert_that` so far worked without ceremony. That is because Beam
*reifies* the global window by default: when `use_global_window=True` (the
default), the elements arrive at the matcher as plain values.

Turn that off — which you must, as soon as your pipeline uses
`beam.WindowInto` — and the elements arrive as `WindowedValue` objects. Each
one carries a value, a timestamp, and a window with a start and an end.

Now compare those objects against plain tuples and watch it break. This is the
real error, not a paraphrase:


In [ ]:
WINDOW_EVENTS = [
    ("2026-01-15T10:05:00+00:00", 10.0),
    ("2026-01-15T10:55:00+00:00", 20.0),
    ("2026-01-15T11:05:00+00:00", 30.0),
]


def _to_epoch(timestamp):
    """Convert an ISO-8601 timestamp to epoch seconds."""
    return datetime.fromisoformat(timestamp).timestamp()


try:
    with TestPipeline(options=TEST_OPTS) as p:
        windowed = (
            p
            | "Create" >> beam.Create(WINDOW_EVENTS)
            | "WithTimestamps" >> beam.Map(
                lambda pair: (Timestamp(_to_epoch(pair[0])), pair[1])
            )
            | "Window" >> beam.WindowInto(FixedWindows(3600))
            | "KeyByRegion" >> beam.Map(lambda pair: ("all", pair[1]))
            | "Count" >> beam.combiners.Count.PerKey()
        )
        # The values ARE right: two events in the 10:00 hour, one in the 11:00 hour.
        assert_that(
            windowed,
            equal_to([("all", 2), ("all", 1)]),
            use_global_window=False,
        )
    print("NO ERROR")
except Exception as exc:
    print("ERROR TYPE:", type(exc).__name__)
    print("ERROR MSG :", exc)


There is the failure, and it is a *confusing* one:

```
AttributeError: 'list' object has no attribute 'end' [while running 'assert_that/Match']
```

The message is about `'list'` and `.end`, and it does not mention windows at
all. Here is what is happening.

`equal_to` sorts the actual elements before comparing them. The actual
elements are `WindowedValue` objects, and `WindowedValue.__lt__` compares
their **window end** times. Beam then tries to sort the *expected* list —
plain tuples — against those `WindowedValue` objects. A tuple has no `.end`,
so Python raises `AttributeError`.

In other words: the sort compares apples with oranges and the error names the
wrong fruit.

**Two ways out:**

1. Compare the windows too, by building the expected values as
   `WindowedValue` objects. Precise, but brittle — you have to hard-code the
   exact window boundaries.
2. **Unwrap the value and compare the payload.** This is what the course test
   suite does, and it is the right default: what the test is really about is
   *how many* firings happened and *what* each contained, not where the window
   boundaries fell.

Let's write that matcher.


In [ ]:
def _windowed_values(expected):
    """Build a matcher that compares the values, not the windows.

    With `use_global_window=False` the elements arrive as `WindowedValue`
    objects. Comparing them directly against plain tuples fails, because
    the window has a start, an end, and a timestamp.

    This matcher unwraps the value and compares only the payload. The
    window boundaries are deliberately not checked: they depend on the
    timestamps in the test data, and what the test is really about is
    *how many* firings happened and *what* each contained.

    Args:
        expected: The expected values, in any order.

    Returns:
        A Beam matcher.
    """

    def _compare(actual):
        unwrapped = [
            item.value if hasattr(item, "value") else item for item in actual
        ]
        return sorted(unwrapped, key=repr) == sorted(expected, key=repr)

    return _compare


with TestPipeline(options=TEST_OPTS) as p:
    windowed = (
        p
        | "Create" >> beam.Create(WINDOW_EVENTS)
        | "WithTimestamps" >> beam.Map(
            lambda pair: (Timestamp(_to_epoch(pair[0])), pair[1])
        )
        | "Window" >> beam.WindowInto(FixedWindows(3600))
        | "KeyByRegion" >> beam.Map(lambda pair: ("all", pair[1]))
        | "Count" >> beam.combiners.Count.PerKey()
    )
    assert_that(
        windowed,
        _windowed_values([("all", 2), ("all", 1)]),
        use_global_window=False,
    )

print("the custom matcher works: two firings, values 2 and 1")


Note one detail in the matcher: `sorted(unwrapped, key=repr)`. Sorting by
`repr` rather than by value means the matcher works for any payload type —
tuples, dicts, dataclasses — without needing a comparison key per type. It
also makes the comparison **order-insensitive**, which is what you want: Beam
does not guarantee the order elements come out of a shuffle.

## 6. Level 4 — testing the whole pipeline

`TestPipeline` can also drive the course's own `build_pipeline` function over
a temporary file. This is the level that catches wiring bugs — a filter in the
wrong place, a sink that writes the wrong columns.

The sink writes a file, so the test reads the file back rather than asserting
on a PCollection.


In [ ]:
import tempfile

from batch_pipeline import build_pipeline

WORK = Path(tempfile.mkdtemp(prefix="dataflow_nb06_"))

input_file = WORK / "orders.csv"
input_file.write_text("\n".join(ORDER_LINES) + "\n", encoding="utf-8")
output_prefix = WORK / "summary"

with TestPipeline(options=TEST_OPTS) as p:
    build_pipeline(
        p,
        input_path=str(input_file),
        output_path=str(output_prefix),
    )

output_files = sorted(f for f in WORK.glob("summary*") if f.is_file())
assert output_files, "the pipeline produced no output file"

contents = "".join(f.read_text(encoding="utf-8") for f in output_files)
print(contents)

assert "region,order_count,revenue_eur" in contents
assert "EMEA,3,401.0" in contents
# The cancelled order is filtered out by the pipeline, so AMER has only one order.
assert "AMER,1,50.0" in contents

print("the whole pipeline produced the expected summary")


Notice the difference between this and the combiner test in section 4:
there, AMER was `2` and `1049.0` (the combiner counted the cancelled order);
here, AMER is `1` and `50.0` (the pipeline filtered it out **before** the
combine). Both are correct — they are testing different layers.

## 7. `TestStream` — deterministic streaming tests

Everything so far has been bounded. Streaming pipelines need to be tested
against **unbounded** input, which normally means a real Pub/Sub topic and a
test that takes minutes and is flaky.

`TestStream` removes both problems. You construct the stream in code: the
elements, their timestamps, and how far the watermark advances. The result is
a test that is **deterministic** — it gives the same answer every time, on
every machine, with no network.


In [ ]:
from apache_beam.testing.test_stream import TestStream

BASE = 1_700_000_000  # a fixed epoch so the test is deterministic


def _at(offset, value):
    """Build a (timestamp, value) pair at a fixed offset from BASE."""
    return Timestamp(BASE + offset), value


with TestPipeline(options=TEST_OPTS) as p:
    stream = (
        p
        | "TestStream" >> TestStream()
        .add_elements([_at(0, "a"), _at(10, "b"), _at(20, "c")])
        .advance_watermark_to(Timestamp(BASE + 60))
        .advance_watermark_to_infinity()
    )
    offsets = (
        stream
        | "Offsets" >> beam.Map(lambda item: int(item[0].seconds()) - BASE)
    )
    assert_that(offsets, equal_to([0, 10, 20]))

print("TestStream produced elements at offsets 0, 10 and 20 seconds.")
print()
print("advance_watermark_to(...)  moves the watermark forward in time.")
print("advance_watermark_to_infinity() tells Beam no more data is coming,")
print("so it flushes every remaining window. Without it, a test pipeline")
print("would hang waiting for a watermark that never arrives.")


### The late-data case

The real payoff of `TestStream` is that you can test **late data**
deterministically: emit an element whose timestamp belongs to a window that
has already closed, and assert exactly what the pipeline does with it.

The course's `project/tests/test_pipelines.py` uses exactly this pattern for
the accumulating and discarding modes:

| Trigger | Mode | Emissions |
|---------|------|-----------|
| `AfterCount(2)` | `ACCUMULATING` | `2, 4, 5` — running totals |
| `AfterCount(2)` | `DISCARDING` | `2, 2, 1` — deltas |

Let's pin both down here too, using the same custom matcher.


In [ ]:
from apache_beam.transforms.trigger import AccumulationMode, AfterCount

FIVE = [(f"2026-01-15T10:0{i}:00+00:00", 1.0) for i in range(5)]


def _stamped(events):
    """Attach a Beam timestamp to each (iso_ts, value) pair."""
    return [(Timestamp(_to_epoch(ts)), value) for ts, value in events]


for mode, expected in (
    (AccumulationMode.ACCUMULATING, [("all", 2), ("all", 4), ("all", 5)]),
    (AccumulationMode.DISCARDING, [("all", 2), ("all", 2), ("all", 1)]),
):
    with TestPipeline(options=TEST_OPTS) as p:
        result = (
            p
            | "Create" >> beam.Create(_stamped(FIVE))
            | "Window" >> beam.WindowInto(
                FixedWindows(3600),
                trigger=AfterCount(2),
                accumulation_mode=mode,
            )
            | "KeyByRegion" >> beam.Map(lambda pair: ("all", pair[1]))
            | "Count" >> beam.combiners.Count.PerKey()
        )
        assert_that(
            result,
            _windowed_values(expected),
            use_global_window=False,
        )
    print(f"{str(mode):<40} -> {expected}")


## 8. What to change for production

| Concern | Notebook version | Production version |
|---------|------------------|--------------------|
| Test runner | `BundleBasedDirectRunner` | Same locally; Dataflow runner for the integration tier |
| Test data | A handful of literal lines | Generated by a seeded generator, plus fixtures per schema version |
| Windows | Unwrapped by a custom matcher | Same matcher, plus explicit `WindowedValue` assertions where the boundary matters |
| Streaming | `TestStream` | `TestStream` plus a staging Pub/Sub topic for the smoke test |
| Coverage | Happy path | Boundary values, malformed input, empty input, late data, skewed keys |
| CI | Runs the pytest suite | Runs the suite on every commit, and a nightly job against production-shaped data |

The rule of thumb from this module: **push assertions as far down the
pyramid as they will go**. A parser bug belongs in a pure-function test that
runs in a millisecond, not in a pipeline test that takes four seconds. The
pipeline tests should be reserved for the wiring that only a runner can check.


## 🔑 Takeaways

- **Use a testing pyramid.** Most tests belong to the pure-logic layer, which needs no runner and runs in milliseconds.
- **Split pure logic out of Beam code on purpose.** `project/transforms.py` has a pure half that runs with no Apache Beam installed — that is what makes fast tests possible.
- **Test a `CombineFn` by hand.** `create_accumulator` / `add_input` / `merge_accumulators` / `extract_output` are plain methods; the property that matters is **merge-order independence**.
- **`TestPipeline` + `assert_that`** runs a real transform graph locally, and fails the test when an assertion does not hold. Use the modern `beam.testing.test_pipeline.TestPipeline`, not the deprecated `PTest`.
- **With `use_global_window=False` the elements are `WindowedValue` objects**, and comparing them to plain tuples raises the real `AttributeError: 'list' object has no attribute 'end'`. The fix is a custom matcher that unwraps `.value` — the one in `project/tests/test_pipelines.py`.
- **Two `assert_that` calls in one pipeline need distinct `label=` values**, or Beam raises `RuntimeError: A transform with label "[6]: assert_that" already exists in the pipeline.`
- **`TestStream` makes streaming tests deterministic** — elements at chosen timestamps and a hand-advanced watermark, with no Pub/Sub topic and no sleeping. Always call `advance_watermark_to_infinity()` so the test pipeline can flush.
